# 01 · 数据收集与面板构建

本 notebook 演示如何把**原始数据**变成研究用的**国家-年份-机构面板**。

## 数据来源

| 来源 | 接口 | 内容 | 是否需要密钥 |
| --- | --- | --- | --- |
| World Bank WDI | 公开 REST API | 增长、通胀、债务、储备、经常账户、汇率 | 否 |
| World Bank WGI | 公开 REST API | 六项治理指标 | 否 |
| IMF WEO | DataMapper API | 增长、通胀、政府债务、经常账户 | 否 |
| BIS | SDMX-JSON | 政策利率、有效汇率、跨境信贷 | 否（默认禁用） |
| FRED | 公开 REST API | 利差、汇率、股指 | **是**（`FRED_API_KEY`） |
| 主权评级 | 用户导入 / 合规页面解析 | 评级、展望、行动日期 | 用户自行确认版权 |

> ⚠️ **评级数据版权声明**：S&P、Moody's、Fitch 的评级历史数据库均为商业授权产品。
> 本项目**不打包**任何完整真实评级数据。本 notebook 默认使用
> `data/sample/ratings_sample.csv`（**合成演示数据**，包含
> `provenance = synthetic_demo_only` 标记列），仅用于跑通流程。
> 若需实证结论，请把自有数据放入 `data/raw/ratings.csv`（列结构见
> `data/raw/ratings_template.csv`），并把 `RATINGS_PATH` 指向它。

## 网络与缓存

所有抓取结果都会缓存到 `data/raw/cache/`，并记录来源 URL 与下载时间。
把 `config.yaml` 中的 `data_sources.offline` 设为 `true` 即可完全离线复现。

In [ ]:
# 让 notebook 在任意工作目录下都能导入 src
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import ensure_directories, get_config, get_path
from src.utils.logging_utils import setup_logging

setup_logging("INFO")
ensure_directories()
CFG = get_config()
print("项目根目录:", PROJECT_ROOT)
print("数据源开关:", {k: v.get("enabled") for k, v in CFG["data_sources"].items() if isinstance(v, dict)})

## 1. 载入评级数据

`load_ratings_csv` 会自动完成：列名标准化 → 机构名称归一 → 评级符号映射到 1-21 分值 → 展望标准化。
映射失败的行不会被丢弃，而是记录在 `rating_map_reason` 列中，便于审计。

In [ ]:
from src.ingest.ratings import load_ratings_csv, load_sample_ratings, validate_imported_ratings

RATINGS_PATH = None  # 换成 "data/raw/ratings.csv" 即可使用自有数据

ratings = load_ratings_csv(RATINGS_PATH) if RATINGS_PATH else load_sample_ratings()
print(f"评级记录: {len(ratings):,} 行")
print("列:", list(ratings.columns))
ratings.head(8)

In [ ]:
# 数据质量检查：映射成功率、机构覆盖、年份连续性
quality = validate_imported_ratings(ratings)
print(quality.to_string(index=False))

unmapped = ratings.loc[ratings["rating_score"].isna(), ["country_iso3", "year", "agency", "rating"]]
print(f"\n未能映射的评级: {len(unmapped)} 条")
unmapped.head()

## 2. 抓取宏观数据（World Bank WDI / WGI）

默认样本国家取自评级数据。`offline=True` 时只读缓存、不发请求。

WGI 指标自 1996 年起发布，早期年份覆盖不全属于**结构性缺失**，
在后续清洗步骤中不会被前向填充（详见 `docs/methodology.md`）。

In [ ]:
from src.ingest.worldbank import fetch_wdi_indicators, fetch_wgi_indicators, to_wide

COUNTRIES = sorted(ratings["country_iso3"].dropna().unique().tolist())
START_YEAR, END_YEAR = CFG["sample"]["start_year"], CFG["sample"]["end_year"]
print(f"国家数: {len(COUNTRIES)} | 年份: {START_YEAR}-{END_YEAR}")

# 离线环境下这两个调用会返回空表并给出提示，不会中断 notebook
wdi_long = fetch_wdi_indicators(countries=COUNTRIES, start_year=START_YEAR, end_year=END_YEAR)
wgi_long = fetch_wgi_indicators(countries=COUNTRIES, start_year=START_YEAR, end_year=END_YEAR)
print(f"WDI 长表: {len(wdi_long):,} 行 | WGI 长表: {len(wgi_long):,} 行")

In [ ]:
# 可选：IMF WEO（接口结构见 src/ingest/imf.py）
from src.ingest.imf import fetch_weo_indicators

weo_long = fetch_weo_indicators(countries=COUNTRIES, start_year=START_YEAR, end_year=END_YEAR)
print(f"IMF WEO 长表: {len(weo_long):,} 行")

# 可选：FRED（需要 FRED_API_KEY 环境变量，切勿硬编码密钥）
from src.ingest.fred import fetch_fred_series, fred_available

print("FRED 密钥已配置:", fred_available())
if fred_available():
    fred_df = fetch_fred_series(["DGS10"], start_date=f"{START_YEAR}-01-01")
    print(f"FRED 观测: {len(fred_df):,} 行")

## 3. 组装国家-年份-机构面板

面板中同时保留两个容易混淆的概念：

* `rating_score` —— 该年**实际发生**的评级（仅在有评级行动的年份非空）
* `score_in_effect` —— 该年**生效**的评级（无行动年份沿用上一次评级）
* `rating_change` —— 生效评级相对上一年的变化，即**评级迁移**的度量

评级迁移矩阵与预测模型都基于 `score_in_effect`/`rating_change`。

In [ ]:
from src.clean.panel import build_country_year_panel, summarize_panel, validate_panel

# 若网络数据不可得，退回使用示例宏观面板（同样是合成数据）
macro_fallback = pd.read_csv(get_path("sample") / "macro_sample.csv", encoding="utf-8-sig")

panel = build_country_year_panel(
    ratings, macro_fallback, start_year=START_YEAR, end_year=END_YEAR, carry_forward=True
)
print(f"面板: {panel.shape[0]:,} 行 × {panel.shape[1]} 列")
validate_panel(panel)

In [ ]:
summarize_panel(panel)

In [ ]:
# 写出中间产物与最终面板
from src.utils.io import write_csv

write_csv(ratings, get_path("interim") / "ratings_mapped.csv")
write_csv(panel, get_path("processed") / "panel_country_year.csv")
print("已写出: data/interim/ratings_mapped.csv 与 data/processed/panel_country_year.csv")